# Семинар 10. Ансамблирование

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import make_classification, fetch_california_housing, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, mean_squared_error

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, RandomForestRegressor, IsolationForest

import scipy.stats as sps
import shap # если не запустится, пишем !pip install shap
import warnings

# sns.set(style="darkgrid", font_scale=1) # можно откомментировать, но не факт, что подойдет для всех графиков
np.random.seed(52)

## План семинара

1. Сравнение деревянных моделей
2. Bias–variance tradeoff
3. Ошибка OOB
4. Feature importance, Shap
5. Isolation Forest
6. Невошедшее. Quantile Regression Forest. XGBoost, LightGBM, CatBoost

## 1. Деревья. Бэггинг. Случайный лес

Сравним эффективность деревянных алгоритмов на синтетическом датасете. Для обучения и генерации будем использовать стандартную библиотеку `sklearn`, в частности метод `make_classification`

In [ ]:
X, y = make_classification(
    n_samples=800, n_features=2,
    n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.0,
    flip_y=0.15
)

Разделим выборку на обучающую и валидационную и визуализируем получившийся датасет

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

fig = plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm')

plt.title("Синтетический датасет")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")

plt.show()

Добавим функцию для визуализации разделяющих поверхностей

In [ ]:
def plot_decision_boundary(model, X, y, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(6, 5))

    plt.contourf(xx, yy, Z, alpha=0.35, cmap='coolwarm')

    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=25, edgecolor='k')
    plt.title(title)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")

    plt.show()

Обучим несколько моделей и сравним

In [ ]:
N_ESTIMATORS = 50

# 1) Одно дерево
tree = DecisionTreeClassifier(max_depth=5)
tree.fit(X_train, y_train)

# 2) Bagging
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=N_ESTIMATORS,
    bootstrap=True
)
bag.fit(X_train, y_train)

# 3) Random Forest
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_features=2
)
rf.fit(X_train, y_train)

models = {
    "Decision Tree": tree,
    "Bagging": bag,
    "Random Forest": rf
}

for name, model in models.items():
    pred = model.predict(X_test)
    print(f"{name:15s} | accuracy = {accuracy_score(y_test, pred):.3f}")

In [ ]:
plot_decision_boundary(tree, X, y, "Decision Tree")

In [ ]:
plot_decision_boundary(bag, X, y, "Bagging")

In [ ]:
plot_decision_boundary(rf, X, y, "Random Forest")

Получилось, что обычное дерево работает лучше чем наши ансамбли. В чем причина?

Давайте перейдем к датасету Титаник. Он содержит побольше признаков, его мы уже подробно обсуждали на предыдущих семинарах. Проведем небольшую предобработку

In [ ]:
# Загружаем Titanic из openml
from sklearn.datasets import fetch_openml
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame

# Оставим простые признаки
df = df[["survived", "pclass", "sex", "age", "fare", "embarked"]].dropna()

X = df.drop("survived", axis=1)
y = df["survived"].astype(int)

# За one-hot энкодим категориальные фичи
categorical = ["sex", "embarked"]
numeric = ["pclass", "age", "fare"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numeric)
])

Обучим несколько ансамблей и посмотрим на качество. В качестве бейзлайна возьмем логистическую регрессию. Дополнительно добавим препроцессинг категориальных фичей и скейлер

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

logreg = Pipeline([
    ("prep", preprocess),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=500))
])

logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)
baseline_score = accuracy_score(y_test, logreg_pred)

scores = []
# Лес из 1 дерева тут можно интерпретировать как решающее дерево
num_trees_list = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500, 1000, 5000]

for n in num_trees_list:
    model = Pipeline([
        ("prep", preprocess),
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=n,
            max_depth=6,
            min_samples_leaf=3,
            max_features="sqrt"
        ))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores.append(accuracy_score(y_test, pred))

plt.figure(figsize=(7, 5))
plt.plot(num_trees_list, scores, marker="o", label="Random Forest")
plt.axhline(baseline_score, linestyle="--", color="red", label="Логистическая регрессия")

plt.title("Качество моделей на Titanic")
plt.xlabel("Количество деревьев (логарифмическая шкала)")
plt.ylabel("Accuracy")

plt.xscale("log")
plt.grid(True)
plt.legend()
plt.show()

Какие тут можно сделать выводы?

## 2. Bias-variance tradeoff

$$
\mathrm{MSE} =
\bigl(f(x) - \mathbb{E}[\hat f(x)]\bigr)^{2}
+ \mathbb{E}\Bigl[\bigl(\mathbb{E}[\hat f(x)] - \hat f(x)\bigr)^{2}\Bigr]
+ \sigma^{2}
$$

$$
= \mathrm{Bias} \left[\hat f(x)\right]^{2}
+ \mathrm{Var} \left[\hat f(x)\right]
+ \sigma^{2}.
$$

Проанализируем разложение ошибки для различных архитектур. Для этого нам пригодится функция, которая будет эмулировать матожидание. Для чистоты эксперимента мы будем рассматривать задачу регрессии, а именно предсказание цен на дома в Калифорнии

*Замечание:* случайность в машинном обучении возникает из множества источников: из тренировочной выборки, из самого распределения данных, случайности в инициализации моделей, алгоритмах обучения и самой архитектуре (например, выбор случайных признаков для леса)

In [ ]:
california_housing = fetch_california_housing()

print(california_housing.DESCR)

X = california_housing.data
y = california_housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
california_housing.data.shape

In [ ]:
def bootstrap_predictions(model_class, model_params,
                          X_train, y_train, X_test, n_runs=10):
    preds = []

    for _ in range(n_runs):
        X_b, y_b = resample(X_train, y_train + sps.norm().rvs([len(X_train)]))

        model = model_class(**model_params)
        model.fit(X_b, y_b)
        preds.append(model.predict(X_test))

    return np.array(preds)

Какие случайности учитывает данная оценка?

Посмотрим на значения bias и variance для различных архитектур



In [ ]:
def bias_variance_decomposition(preds, y_test):
    mean_pred = np.mean(preds, axis=0)

    bias2 = np.mean((mean_pred - y_test) ** 2)
    variance = np.mean(np.var(preds, axis=0))

    mse_total = np.mean((preds - y_test) ** 2)
    noise = mse_total - bias2 - variance

    return bias2, variance, noise

In [ ]:
models = [
    ("Linear Regression", LinearRegression, {}),
    ("Ridge Regression", Ridge, {"alpha": 1}),
    ("Decision Tree", DecisionTreeRegressor, {}),
    ("Random Forest", RandomForestRegressor, {"n_estimators": 50}),
]

results = []

for name, model_class, params in models:
    preds = bootstrap_predictions(model_class, params, X_train, y_train, X_test)
    bias2, var, noise = bias_variance_decomposition(preds, y_test)

    results.append((name, bias2, var, noise))

Посмотрим какой порядок значений лосса стоит ожидать от каждого из методов

In [ ]:
for name, model, args in models:
    model = model(**args)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name:15s} | MSE = {mean_squared_error(y_test, pred):.3f}")

Тут важно обратить внимание на то что изначально цена указана в сотнях тысяч долларов, иначе эта цифра не интерпретируема

In [ ]:
y_train.mean()

In [ ]:
names = [r[0] for r in results]
bias2 = np.array([r[1] for r in results])
var = np.array([r[2] for r in results])
noise = np.array([r[3] for r in results])

x = np.arange(len(names))

width = 0.35
plt.figure(figsize=(12, 6))

# plt.bar(x - width / 2, bias2, width, label="$Bias^2$")
# plt.bar(x + width / 2, var, width, label="Variance")

plt.bar(x - width, bias2, width, label="$Bias^2$")
plt.bar(x, var, width, label="Variance")
plt.bar(x + width, noise, width, label="Noise")

plt.xticks(x, names)
plt.ylabel("Value")
plt.title("Bias–Variance Decomposition")

plt.legend()
plt.grid(axis="y", alpha=0.3)
# plt.yscale("log")
plt.show()

Куда делся шум? Какие выводы можно сделать о моделях на основе этих результатов? Как это обосновывается архитектурой моделей и алгоритмом их обучения?

## 3. Ошибка OOB

При использовании метода бутстрапа для построения ансамблей, можно приблизительно посчитать, какая часть уникальных данных попадает в выборку для обучение каждой модели


Пусть в исходной выборке $n$ объектов. На каждом шаге формирования подвыборки мы вытаскиваем случайный объект, возвращаем его обратно и повторяем процедуру $n$ раз

1.  Вероятность выбрать конкретный объект на одном шаге равна $\frac{1}{n}$.
2.  Соответственно, вероятность не выбрать этот объект на одном шаге: $1 - \frac{1}{n}$.
3.  Так как выборка формируется $n$ раз, вероятность того, что конкретный объект $x$ ни разу не попадет в подвыборку $X_i$ ($|X_i| = |X| = n$):

$$P(x \notin X) = \left(1 - \frac{1}{n}\right)^n$$

При увеличении размера выборки ($n \rightarrow +\infty$) это выражение стремится к второму замечательному пределу:

$$\lim_{n \rightarrow +\infty} \left(1 - \frac{1}{n} \right)^n = \frac{1}{e} \approx 0.37$$

В среднем, каждая базовая модель обучается лишь на **~63%** уникальных объектов исходной выборки. Оставшиеся **~37%** (OOB данные) можно использовать для валидации модели без необходимости откладывать отдельную тестовую выборку

In [ ]:
n = 10000

# выборка с возвращением того же размера
bootstrap_sample = np.random.choice(n, size=n, replace=True)

unique = len(np.unique(bootstrap_sample))

ratio = unique / n

print(f"Доля уникальных объектов: {ratio:.5f}")
print(f"Теоретический предел (1 - 1/e): {1 - 1/np.e:.5f}")

## 4. Feature Importance

Деревянные модели позволяют оценивать важность признаков в дереве решений. Каким образом можно это осуществлять?

1.  В каждом узле дерева выбирается признак, который наилучшим образом разделяет данные, максимизируя выбранный критерий информации

2.  Для каждого признака вычисляется суммарное улучшение этого критерия во всех узлах, где он использовался. При этом каждое улучшение взвешивается на количество обучающих образцов, которые прошли через соответствующий узел

3.  Итоговые взвешенные суммы для всех признаков нормализуются так, чтобы их общая сумма равнялась 1. Так мы получаем важность каждого признака для одного дерева

*Замечание:* для ансамбля важности дополнительно усредняются по всем деревьям в ансамбле


Очень удобный метод `make_classification` позволяет создавать неинформативные признаки. На основе синтетического датасета с полезными, дублирующимися и шумовыми признаками продемонстрируем механизм отбора признаков

In [ ]:
# 1000 объектов, 10 признаков: 3 полезных, 2 дублирующихся, 5 шумовых.
X, y = make_classification(n_samples=1000,
                           n_features=10,
                           n_informative=3,
                           n_redundant=2,
                           shuffle=False)

feature_names = ["$x_{" + f"{i + 1}" + "}$" for i in range(10)]

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X, y)

importances = rf.feature_importances_

df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', hue='feature', data=df_importance, palette='husl', legend=False)

plt.title('Feature Importance')
plt.xlabel('Средняя информативнсоть')
plt.ylabel('Признаки')
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.show()
print(f"Сумма важностей суммируется в 1: {np.sum(importances):.2f}")

Зачем нужно считать важность признаков?

Feature Selection позволяет отбирать признаки на основе их информативности, для этого существуют различные подходы, можно выделить основные:

1. Фильтрационные на основе статистических критериев
2. Встроенные на основе архитектуры
3. Black-Box методы

## SHAP (SHapley Additive exPlanations)



SHAP — это метод интерпретации результатов моделей машинного обучения, основанный на теории игр и концепции векторов Шепли. Этот подход позволяет объяснить выход модели как сумму вкладов каждого отдельного признака

Основная идея заключается в том, что прогноз модели для конкретного объекта рассматривается как "выигрыш" в кооперативной игре, а признаки объекта — как "игроки". Значение Шепли для каждого признака рассчитывается путем усреднения его вкладов во все возможные комбинации других признаков

Ключевые особенности метода:

1. Локальная интерпретируемость: SHAP объясняет, почему модель сделала именно такое предсказание для конкретного наблюдения
2. Аддитивность: сумма значений SHAP для всех признаков плюс базовое значение (среднее предсказание по выборке) равна фактическому предсказанию модели
3. Глобальная важность: усредняя абсолютные значения SHAP по всей выборке, можно получить оценку глобальной важности признаков, которая часто более точна, чем стандартные методы оценки важности в деревьях

In [ ]:
warnings.filterwarnings('ignore') # в целях наглядности, на практике так стараемся не делать

shap.initjs()

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

shap_values_class_1 = shap_values[:, :, 1]

shap.summary_plot(shap_values_class_1, X, feature_names=feature_names)
shap.summary_plot(shap_values_class_1, X, feature_names=feature_names, plot_type="bar")

Можете самостоятельно ознакомиться с фунционалом шапа в [их документации](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html) (не самая удобная вещь, но лучше чем ничего)

## 5. Isolation Forest

Деревья и ансамбли хотя и могут показаться идеей устаревшей, по-сравнению с текущими SOTA методами, но они используются в многих методах и нестандартных задачах до сих пор

Isolation Forest представляет собой ансамбль *очень случайных деревьев*, задача которых не в предсказании целевой переменной, а в обработке пространства признаков. Каждое дерево в ансамбле - случайное

### Алгоритм построения дерева изоляции (iTree)

Пусть у нас есть подвыборка объектов (можно брать размер порядка 256)

Повторяем рекурсивно алгоритм, напоминающий стандартный процесс построения дерева:


1. Выбирается случайный признак
$$
i \sim U(1, d).
$$

2. Выбирается случайный порог из текущего диапазона значений признака
$$
t \sim U(x_i^{\min}, x_i^{\max}).
$$

3. Вершина становится листом, если:
   * достигнута максимальная глубина
   * в узле только один объект
   * все объекты имеют одинаковые значения по всем признакам


### Anomaly score

После построения ансамбля деревьев

1. Для каждого обьекта вычисляем среднюю глубину его листа, усреднённую по деревьям

2. Нормируем глубину с использованием гармонического числа $H(k) = 1 + \frac{1}{2} + \ldots + \frac{1}{k}:$

$$
c(m) =
\begin{cases}
2H(m - 1) - \dfrac{2(m - 1)}{n} & m > 2, \\
1 & m = 2, \\
0 & \text{otherwise},
\end{cases}
$$

где $m$ - количество обьектов на одно дерево, $n$ - колчество деревьев.

$c(m)$ это матож средней глубины (б/д)

3. Финальный anomaly score:
$$
s(x) = 2^{-\frac{h(x)}{c(m)}}.
$$

   * Если $h(x)$ маленькая, то объект легко изолируется, $s(x) > 0.5$
   * Если $h(x)$ большая, то объект трудно изолируется, $s(x) < 0.5$

In [ ]:
N_POINTS = 300
N_ANOMALIES = 40

# кластер
X, _ = make_blobs(
    n_samples=N_POINTS,
    centers=[[0, 0]],
    cluster_std=0.6
)

# outliers
outliers = sps.uniform.rvs(loc=-6, scale=12, size=(N_ANOMALIES, 2))

X_all = np.vstack([X, outliers])


iso = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=0.07,
)
iso.fit(X_all)

pred = iso.predict(X_all) # предсказания
scores = iso.decision_function(X_all) # сами скоры

plt.figure(figsize=(6, 6))
mask_anom = (pred == -1)

plt.scatter(
    X_all[~mask_anom, 0],
    X_all[~mask_anom, 1],
    s=25,
    alpha=0.8,
    label="normal"
)

plt.scatter(
    X_all[mask_anom, 0],
    X_all[mask_anom, 1],
    s=40,
    marker='x',
    alpha=0.9,
    label="anomaly"
)

plt.title("Результаты детекции")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.grid(True)
plt.legend()
plt.show()

print("Всего точек:", len(X_all))
print("Найдено аномалий:", (pred == -1).sum())
print("Диапазон decision_function: [{:.3f}, {:.3f}]".format(scores.min(), scores.max()))

Имплементации могут различаться, поэтому механизм библиотеки `sklearn` дает другой диапазон скоров

In [ ]:
# Карта уровней anomaly scores
xx, yy = np.meshgrid(
    np.linspace(X_all[:, 0].min() - 1, X_all[:, 0].max() + 1, 300),
    np.linspace(X_all[:, 1].min() - 1, X_all[:, 1].max() + 1, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = iso.decision_function(grid).reshape(xx.shape)

plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, Z, levels=20, alpha=0.8)
plt.scatter(X_all[:, 0], X_all[:, 1], s=12, alpha=0.8)

plt.title("Карта уровней anomaly scores")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.grid(True)
plt.show()

# Гистограмма anomaly scores
plt.figure(figsize=(6, 4))
plt.hist(scores, bins=35, alpha=0.9)
plt.title("Гистограмма anomaly scores")
plt.xlabel("anomaly score (больше - нормальнее)")
plt.ylabel("Количество точек")
plt.grid(True)
plt.show()

Про Isolation Forest неплохая [статья](https://en.wikipedia.org/wiki/Isolation_forest) на википедии. Вариант дольше, но интересней - прочитать оригинальную статью [тут](https://ieeexplore.ieee.org/document/4781136)

## 6.  Невошедшее. Quantile Regression Forest. XGBoost, LightGBM, CatBoost

Существует множество приложений деревьев и ансамблей и не все ансамбли строятся из деревьев

В задаче регрессии из данных в листьях можно доставать не только усредненные и медианные значения, но и использовать дополнительную информацию о распределении данных, чтобы строить доверительные интервалы и предсказывать различные статистики (см. [Quantile Regression Forest](https://www.jmlr.org/papers/volume7/meinshausen06a/meinshausen06a.pdf))

При помощи ансамблей можно эффективно оценивать неопределенность моделей, находить данные приходящие из незнакомых распределений (см. [Uncertainty Quantification](https://en.wikipedia.org/wiki/Uncertainty_quantification))

Деревья используются как строительная единица для бустинга - SOTA метода для решения задачи классификации (см. следующие лекции)

Следующая эпоха после стандартных ансамблевых методов (по типу бэггинга и леса) приходится на градиентный бустинг. Бустинг это один из SOTA методов на текущий день (хотя сам он появился еще очень давно). Основные библиотеки, его реализующие это `XGBoost`, `LightGBM`, `CatBoost`. Про эти библиотеки вы подробнее узнаете на будущих семинарах. Про них упомянуто с целью напомнить о том, что `scikit-learn` подходит только для учебных целей, даже для обучения регрессии есть более специализированные библиотеки, например `statsmodels`